<a href="https://colab.research.google.com/github/Abdul03Khader/Fake-News-Detection-NLP/blob/main/FakeAndRealNewsPrediction(NLP).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
fake = pd.read_csv("/content/Fake.csv")
real = pd.read_csv("/content/True.csv")

In [26]:
fake['label'] = 0
real['label'] = 1

In [27]:
#concatination of both fake and real news
df = pd.concat([fake,real],axis=0)

In [28]:
#shuffling the data Standard step after the combination of two datsets
df = df.sample(frac=1).reset_index(drop=True)

In [29]:
df.head()

,title,text,subject,date,label
0,Protests planned for Trump speech at pro-Israe...,NEW YORK (Reuters) - Some rabbis and Jewish st...,politicsNews,"March 17, 2016",1
1,Zuckerberg again rejects claims of Facebook im...,NEW YORK (Reuters) - Facebook Inc chief execut...,politicsNews,"November 13, 2016",1
2,Putin links Japan peace treaty to Tokyo's alli...,"DANANG, Vietnam (Reuters) - Concluding a peace...",worldnews,"November 11, 2017",1
3,FLASHBACK: GRAPHIC VIDEO SHOWS Hillary Support...,Did Hillary forget about her deplorable supp...,left-news,"Sep 11, 2016",0
4,Trump to offer federal coal to industry awash ...,WASHINGTON (Reuters) - U.S. President Donald ...,politicsNews,"March 28, 2017",1


In [30]:
df.shape

(44898, 5)

In [31]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44898 entries, 0 to 44897
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   title    44898 non-null  object
 1   text     44898 non-null  object
 2   subject  44898 non-null  object
 3   date     44898 non-null  object
 4   label    44898 non-null  int64 
dtypes: int64(1), object(4)
memory usage: 1.7+ MB


In [32]:
df['label'].value_counts()

,count
label,
0,23481
1,21417


In [33]:
df.isna().sum()

,0
title,0
text,0
subject,0
date,0
label,0


In [34]:
#selecting important columns
df = df[['text','label']]

In [35]:
df.head()

,text,label
0,NEW YORK (Reuters) - Some rabbis and Jewish st...,1
1,NEW YORK (Reuters) - Facebook Inc chief execut...,1
2,"DANANG, Vietnam (Reuters) - Concluding a peace...",1
3,Did Hillary forget about her deplorable supp...,0
4,WASHINGTON (Reuters) - U.S. President Donald ...,1


In [36]:
df.isnull().sum()

,0
text,0
label,0


In [37]:
#Text cleaning
import re #regex (text cleaning tool)


#cleaning function
def clean(text):
  text = text.lower()
  text = re.sub(r'[^a-zA-z]', ' ' , text)
  return text


In [38]:
#aplying clean
df['text'] = df['text'].apply(clean)

In [39]:

#Convert text → numbers (TF-IDF)
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

#x and y variable
x = vectorizer.fit_transform(df['text'])
y = df['label']



In [40]:
#train-test-split
from sklearn.model_selection import train_test_split

x_train, x_test, y_train , y_test= train_test_split(x,y,test_size=0.2,random_state=2)


In [41]:
#model -> logistic regression
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()
model.fit(x_train,y_train)

LogisticRegression()

In [42]:
#Accuracy
from sklearn.metrics import accuracy_score

y_pred = model.predict(x_test)
print("Accuracy :",accuracy_score(y_test,y_pred))

Accuracy : 0.9867483296213808


In [43]:
#confusion matrix
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test,y_pred)

print(cm)

[[4601   70]
 [  49 4260]]


In [44]:
#classification report
from sklearn.metrics import classification_report

print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.99      0.99      0.99      4671
           1       0.98      0.99      0.99      4309

    accuracy                           0.99      8980
   macro avg       0.99      0.99      0.99      8980
weighted avg       0.99      0.99      0.99      8980



In [45]:
#pickle
import pickle

pickle.dump(model,open('model.pkl','wb'))

In [46]:
import gradio as gr
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer

# Load the trained model
loaded_model = pickle.load(open('model.pkl', 'rb'))

# Re-initialize and fit the TfidfVectorizer
# Assuming 'df' and 'clean' function are available from previous cells.
# If 'df' is not available, you would need to re-run the data loading and preprocessing steps.
vectorizer = TfidfVectorizer(max_features=5000)
# Fit transform on the cleaned text column of the dataframe
# This recreates the vocabulary and IDF values used during training.
vectorizer.fit_transform(df['text'])

def predict_news(news_text):
    # Clean the input text using the previously defined 'clean' function
    cleaned_text = clean(news_text)
    # Transform the cleaned text using the fitted vectorizer
    vectorized_text = vectorizer.transform([cleaned_text])
    # Make a prediction using the loaded model
    prediction = loaded_model.predict(vectorized_text)

    if prediction[0] == 0:
        return "Fake News"
    else:
        return "Real News"

# Create and launch the Gradio interface
iface = gr.Interface(fn=predict_news, inputs=gr.Textbox(lines=5, label="Enter News Article Text"), outputs="text", title="Fake News Detector")
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://94a2416d3e6f9519d4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
